# MVP graph pack: one JSON for every surface (replayable)

`site/mvp/mvp.json` is read by the web page, the `baebugs-graph` CLI, the MCP server, `npx github:biobitworks/baebugs`
and the Claude skill, so they all answer with the same root. Levels: root -> module -> leaf, with bug / species / part /
labware / result views pointing at their leaf. **Swap data:** set `PACK` to another pack (a fork's, a BYO build's, or
`None` for the published copy on `main`) and `REPO` to the tree you want to check it against.

In [1]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == 'notebooks': os.chdir('..')
sys.path.insert(0, os.getcwd())
import subprocess, shutil
import pandas as pd
from baebugs import mvp, aok
from baebugs.explore import merkle_root

REPO = '.'                     # swap: tree whose bytes the pack must resolve to
PACK = 'site/mvp/mvp.json'     # swap: any pack path, or None -> repo copy, else published copy on main
pack = mvp.load(PACK)
print('root', pack['root'], '|', len(pack['leaves']), 'leaves |', len(pack['bugs']), 'bugs |', len(pack['species']), 'species |',
      len(pack['parts']), 'parts |', len(pack['labware']), 'labware')

root 7e3150f9a44bf196ad96b5345c3130cea2921caa43869bf52eca6fec0bd271b1 | 741 leaves | 91 bugs | 278 species | 25 parts | 4 labware


## The root is a Merkle root of the AoK root, the hypergraph root and every MMR log root

In [2]:
assert merkle_root(sorted(v for v in pack['roots'].values() if v)) == pack['root']
pd.DataFrame([{'log': k, 'root': (v or '')[:16], **({'n': pack['mmr'][k]['n'], 'verified': pack['mmr'][k]['verified']} if k in pack['mmr'] else {})}
              for k, v in pack['roots'].items()])

,log,root,n,verified
0,aok,396a0d2e4f86fcf9,NaN,NaN
1,hybrids_mmr,d7e422523f408c84,3.0,True
2,hypergraph,8cc197054b823ec7,NaN,NaN
3,hypergraph_mmr,c20a893a164051e5,4.0,True
4,impact_mmr,125c3fc63a3be772,104.0,True
5,labbox_mmr,234a9637a12b23e7,1.0,True
6,upstream_mmr,38a4a379f5c6d0d5,12.0,True


## Addresses: root, module, leaf, sha, or free text

In [3]:
for q in ('root', 'module:lab-parts', 'leaf:LICENSE', 'Apidae'):
    r = mvp.get(pack, q)
    print(q, '->', json.dumps({k: (v[:6] if isinstance(v, list) else v) for k, v in r.items() if k not in ('roots',)})[:220])

root -> {"address": "root", "root": "7e3150f9a44bf196ad96b5345c3130cea2921caa43869bf52eca6fec0bd271b1"}
module:lab-parts -> {"address": "module:lab-parts", "atoms": 106, "claims": {"OWN": 106}, "id": "module:lab-parts", "root": "a974e369a4d04726a4c9c501332d27f33a0c4598c977ac3985fb1e0589ee3430", "status": {"built": 106}, "terms": [], "leaves":
leaf:LICENSE -> {"address": "leaf:LICENSE", "claim": "OWN", "kind": "meta", "leaf": "89764acbe9dd5093cd6c9ad642427eb1e34aae45b11e70c698f3d0791b1d5083", "licence": "MIT", "module": "repo-meta", "path": "LICENSE", "sha256": "2460f3bbefc63
Apidae -> {"query": "Apidae", "hits": ["leaf:print/fuse1/california/Apidae_mean+1SD_17.1mm.stl", "leaf:print/fuse1/california/Apidae_mean+2SD_20.7mm.stl", "leaf:print/fuse1/california/Apidae_mean-1SD_9.9mm.stl", "leaf:print/fuse1/


## Resolvability: every hash the pack references must resolve to stored bytes
A leaf whose file is missing or whose bytes differ is a BREAK, not a silent skip.

In [4]:
from collections import Counter
v = Counter(); bad = []
for x in pack['leaves']:
    p = Path(REPO, x['path'])
    s = 'MISSING' if not p.exists() else 'MATCH' if aok._sha(p) == x['sha256'] else 'DIFFERS'
    v[s] += 1
    if s != 'MATCH': bad.append((s, x['path']))
print(dict(v)); bad[:10]

{'MATCH': 736, 'DIFFERS': 5}


[('DIFFERS', 'notebooks/aok.ipynb'),
 ('DIFFERS', 'notebooks/compare_byo.ipynb'),
 ('DIFFERS', 'notebooks/mvp_graph_pack.ipynb'),
 ('DIFFERS', 'notebooks/taxon_crops_004.ipynb'),
 ('DIFFERS', 'notebooks/watch_commit.ipynb')]

## Custody breaks are recorded, not hidden

In [5]:
print(pack['custody']['breaks'], 'breaks |', pack['custody']['rule'])
pd.DataFrame(pack['results'])[['protocol', 'path', 'runs', 'custody', 'statuses']]

8 breaks | every hash the graph references must resolve to stored bytes; otherwise BREAK


,protocol,path,runs,custody,statuses
0,DAISY-BB-001b,daisy/results/DAISY-BB-001b/results.json,6,NO WEIGHTS (runs failed),[FAILED]
1,DAISY-BB-001b,daisy/results/DAISY-BB-001b/v6/results.json,6,BROKEN (weights not stored),[OBSERVED_EXPLORATORY]
2,DAISY-BB-002a,daisy/results/DAISY-BB-002a/v2/results.json,2,BROKEN (weights not stored),[OBSERVED_EXPLORATORY]
3,DAISY-BB-004,daisy/results/DAISY-BB-004/v4/results.json,3,INTACT,[OBSERVED_EXPLORATORY]


## Bring your own: make an FCO of files you hold, prove it against the graph (same as the web page)

In [6]:
fco = mvp.fco_of(['LICENSE', 'README.md'], REPO)
fco['leaves'].append({'path': 'print/fuse1/manifest.csv', 'sha256': '0' * 64})   # a tampered copy
att = mvp.prove(pack, fco)
print(att['match'], 'of', att['compared'], 'match graph', att['graph_root'][:12])
[(r['path'], r['verdict']) for r in att['rows']]

2 of 3 match graph 7e3150f9a44b


[('LICENSE', 'MATCH'),
 ('README.md', 'MATCH'),
 ('print/fuse1/manifest.csv', 'DIFFERS')]

## Views: bugs (printable toys), species, lab parts

In [7]:
pd.DataFrame(pack['bugs'])[['order', 'family', 'level', 'length_mm', 'kit_tray', 'path']].head(10)

,order,family,level,length_mm,kit_tray,path
0,Coleoptera,Coccinellidae,mean-2SD,3.7,HEX-2,print/fuse1/california/Coccinellidae_mean-2SD_...
1,Coleoptera,Coccinellidae,mean-1SD,5.0,HEX-2,print/fuse1/california/Coccinellidae_mean-1SD_...
2,Coleoptera,Coccinellidae,mean,6.3,GRID-4,print/fuse1/california/Coccinellidae_mean_6.3m...
3,Coleoptera,Coccinellidae,mean+1SD,7.6,PERF-5,print/fuse1/california/Coccinellidae_mean+1SD_...
4,Coleoptera,Coccinellidae,mean+2SD,8.9,PERF-5,print/fuse1/california/Coccinellidae_mean+2SD_...
5,Hymenoptera,Formicidae,mean-2SD,2.5,TRAY,print/fuse1/california/Formicidae_mean-2SD_2.5...
6,Hymenoptera,Formicidae,mean-1SD,4.7,TRAY,print/fuse1/california/Formicidae_mean-1SD_4.7...
7,Hymenoptera,Formicidae,mean,6.9,SCREEN-1,print/fuse1/california/Formicidae_mean_6.9mm.stl
8,Hymenoptera,Formicidae,mean+1SD,9.1,SCREEN-1,print/fuse1/california/Formicidae_mean+1SD_9.1...
9,Hymenoptera,Formicidae,mean+2SD,11.3,HEX-2,print/fuse1/california/Formicidae_mean+2SD_11....


In [8]:
pd.DataFrame(pack['species']).sort_values('n', ascending=False)[['order', 'family', 'species', 'n', 'length_mm_median', 'licences']].head(10)

,order,family,species,n,length_mm_median,licences
2,Coleoptera,?,Coleoptera,270,14.4,[zero/1.0/legalcode]
40,Diptera,Bombyliidae,"Conophorus fenestratus (Osten Sacken, 1877)",173,13.2,[zero/1.0/legalcode]
37,Diptera,Bombyliidae,"Bombylius Linnaeus, 1758",119,17.2,[zero/1.0/legalcode]
183,Hymenoptera,Halictidae,"Lasioglossum Curtis, 1833",97,9.6,[zero/1.0/legalcode]
91,Hymenoptera,?,Hymenoptera,80,9.4,[zero/1.0/legalcode]
95,Hymenoptera,Andrenidae,"Andrena candida Smith, 1879",77,8.8,[zero/1.0/legalcode]
230,Lepidoptera,?,Lepidoptera,76,21.2,[zero/1.0/legalcode]
148,Hymenoptera,Apidae,"Melissodes Latreille, 1829",62,13.1,[zero/1.0/legalcode]
33,Diptera,?,Diptera,61,8.3,[zero/1.0/legalcode]
13,Coleoptera,Coccinellidae,Coccinellidae,55,6.1,[zero/1.0/legalcode]


In [9]:
pd.DataFrame(pack['parts'])[['name', 'platform', 'status', 'bbox_mm', 'path']].head(10)

,name,platform,status,bbox_mm,path
0,trommel_zone_2mm,bench,CONCEPT,"[110.0, 110.0, 76.0]",site/lab/stl/trommel_zone_2mm.stl
1,trommel_zone_4mm,bench,CONCEPT,"[110.0, 110.0, 76.0]",site/lab/stl/trommel_zone_4mm.stl
2,trommel_zone_7mm,bench,CONCEPT,"[110.0, 110.0, 76.0]",site/lab/stl/trommel_zone_7mm.stl
3,trommel_cap_inlet,bench,CONCEPT,"[114.0, 114.0, 22.0]",site/lab/stl/trommel_cap_inlet.stl
4,trommel_cap_outlet,bench,CONCEPT,"[114.0, 114.0, 22.0]",site/lab/stl/trommel_cap_outlet.stl
5,trommel_stand_inlet,bench,CONCEPT,"[90.0, 60.0, 134.1]",site/lab/stl/trommel_stand_inlet.stl
6,trommel_stand_outlet,bench,CONCEPT,"[90.0, 60.0, 122.0]",site/lab/stl/trommel_stand_outlet.stl
7,trommel_crank_hand,bench,CONCEPT,"[78.0, 22.0, 47.0]",site/lab/stl/trommel_crank_hand.stl
8,trommel_crank_arm,bench,CONCEPT,"[81.0, 26.0, 47.0]",site/lab/stl/trommel_crank_arm.stl
9,trommel_bin,bench,CONCEPT,"[70.0, 96.0, 40.0]",site/lab/stl/trommel_bin.stl


## Every surface gives the same answer: Python, CLI and (if node is installed) the npx bin

In [10]:
# the CLI and npx bin read the repo pack (site/mvp/mvp.json), so compare roots, not only one leaf
def surface(q):
    out = {'python': mvp.get(pack, q),
           'cli': json.loads(subprocess.run([sys.executable, '-m', 'baebugs.mvp', 'get', q], capture_output=True, text=True, check=True).stdout)}
    if shutil.which('node'):
        out['npx bin'] = json.loads(subprocess.run(['node', 'bin/baebugs.mjs', 'get', q], capture_output=True, text=True, check=True).stdout)
    return out
roots = {k: v['root'] for k, v in surface('root').items()}
lic = surface('leaf:LICENSE')
print({k: r[:16] for k, r in roots.items()})
print('same root on every surface:', len(set(roots.values())) == 1, '| same LICENSE leaf:', all(v == lic['python'] for v in lic.values()))
if len(set(roots.values())) > 1:
    print('PACK is not the repo pack the CLI/npx read; copy it to site/mvp/mvp.json to exercise those surfaces on it')

{'python': '7e3150f9a44bf196', 'cli': '7e3150f9a44bf196', 'npx bin': '7e3150f9a44bf196'}
same root on every surface: True | same LICENSE leaf: True


## Rebuild from the tree
`mvp.build` regenerates the pack in memory. If the root differs from the loaded pack, the tree moved since the pack
was written; `python -m baebugs.mvp` writes a fresh one.

In [11]:
fresh = mvp.build(REPO)
print('fresh', fresh['root'][:16], '| loaded', pack['root'][:16], '| same' if fresh['root'] == pack['root'] else '| tree moved since pack was written')

fresh 9734b5936adfdbce | loaded 7e3150f9a44bf196 | tree moved since pack was written
